# Training Loop


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import torch
from course_tools import CharTokenizer, TinyConfig, TinyTransformerLM, train_model, evaluate_model

LECTURE_NOTE_TITLE = 'Training Loop'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Training Loop


## Forward backward step repeat


In [2]:
text = 'training loops turn data and gradients into learned weights.\n' * 8
tokenizer = CharTokenizer.build([text])
model = TinyTransformerLM(TinyConfig(vocab_size=len(tokenizer.stoi), block_size=32, d_model=32, n_heads=4, n_layers=1, d_ff=64))
history = train_model(model, tokenizer, train_text=text, eval_text=text, steps=10, learning_rate=3e-3, batch_size=4)
print(history)


[{'step': 1.0, 'train_loss': 19.476070404052734, 'eval_loss': 18.776033401489258, 'bpb': 27.08809027589409}, {'step': 2.0, 'train_loss': 18.514345169067383, 'eval_loss': 18.370649337768555, 'bpb': 26.503244697508816}, {'step': 4.0, 'train_loss': 17.794401168823242, 'eval_loss': 17.178550720214844, 'bpb': 24.783409933713486}, {'step': 6.0, 'train_loss': 16.594812393188477, 'eval_loss': 15.924201011657715, 'bpb': 22.9737658296376}, {'step': 8.0, 'train_loss': 15.0789213180542, 'eval_loss': 13.889232635498047, 'bpb': 20.03792704498618}, {'step': 10.0, 'train_loss': 13.766950607299805, 'eval_loss': 13.306866645812988, 'bpb': 19.197750519685155}]


## Validation belongs inside the run


In [3]:
metrics = evaluate_model(model, tokenizer, text)
print(metrics)


{'loss': 13.347046852111816, 'bpb': 19.25571830405437}


## Batch size and accumulation change effective tokens per update


In [4]:
seq_len = model.config.block_size
for micro_batch, accumulation in [(4, 1), (4, 4), (8, 2)]:
    tokens_per_update = micro_batch * accumulation * seq_len
    print({'micro_batch': micro_batch, 'accumulation': accumulation, 'tokens_per_update': tokens_per_update})


{'micro_batch': 4, 'accumulation': 1, 'tokens_per_update': 128}
{'micro_batch': 4, 'accumulation': 4, 'tokens_per_update': 512}
{'micro_batch': 8, 'accumulation': 2, 'tokens_per_update': 512}


## Checkpoints and reports are part of the loop


The runnable script for this lecture writes a checkpoint and a report under artifacts/training_loop_demo/.


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_train.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/base_train.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/checkpoint_manager.py
